# 05｜Patch Merging：Swin 怎样逐层缩小特征图

前面的 W-MSA 和 SW-MSA 只更新 token 内容，不改变 token 数量。

但 Swin 还需要像 CNN 一样逐层缩小空间尺寸，让深层 token 表示更大的图片区域。完成这项工作的模块叫 Patch Merging。

## 1. 为什么需要下采样

如果从第一层到最后一层始终保留 56 × 56 个 tokens，会有两个问题：

1. 深层一直处理大量空间位置，计算量较大。
2. 模型缺少从局部细节到整体语义的层级变化。

CNN 通常通过池化或步长卷积降低高和宽。Swin 使用 Patch Merging 完成类似工作。

## 2. Patch Merging 是什么

Patch Merging 的意思是“合并相邻 patch tokens”。

它每次取空间上相邻的 2 × 2 个 tokens，把四者的信息组合成一个新 token。

因此，原来的四个空间位置会变成一个位置：

- 高度变成原来的一半；
- 宽度变成原来的一半；
- token 总数变成原来的四分之一；
- 每个新 token 对应的原图区域变大。

## 3. 用四个 tokens 看合并过程

假设一个 2 × 2 区域包含：

| 左上 | 右上 |
|---|---|
| token 1 | token 2 |
| token 3 | token 4 |

每个 token 都有 C 个特征数字。

第一步不是求平均，也不是选最大值，而是把四个 token 的特征依次拼接。拼接后，一个位置暂时拥有 4C 个特征。

所以 Patch Merging 会保留四个位置的信息来源，而不是像最大池化那样只保留其中一个最大值。

![Patch Merging 合并过程](images/06_patch_merging.svg)

## 4. 为什么拼接后还要线性映射

四个 C 维 token 拼接后得到 4C 维。如果直接保留 4C，通道数每经过一次合并就扩大四倍，模型会迅速变得很大。

因此 Swin 使用一个可学习的线性层，把 4C 维压缩成 2C 维。

可以理解成：

1. 先把四个邻居的信息全部收集起来；
2. 再学习怎样从 4C 个特征中提炼出 2C 个新特征。

这里的线性层会参与训练，不是人工规定怎样压缩。

## 5. 完整 shape 变化

| 步骤 | shape | 含义 |
|---|---|---|
| 输入 | B × H × W × C | 原 token 网格 |
| 取 2 × 2 邻域并拼接 | B × H/2 × W/2 × 4C | 四个 token 合成一个位置 |
| LayerNorm | B × H/2 × W/2 × 4C | 稳定拼接后的特征分布 |
| 线性映射 | B × H/2 × W/2 × 2C | 把通道从 4C 压到 2C |

LayerNorm 不改变 shape，只调整特征数值。真正改变通道数的是最后的线性映射。

## 6. Stage 1 到 Stage 2 的真实例子

Stage 1 输出为 B × 56 × 56 × 96。

Patch Merging 先把相邻 2 × 2 tokens 拼接：

- 56 × 56 变成 28 × 28；
- 96 维的四个 token 拼成 384 维。

然后线性层把 384 维压缩成 192 维。最终输出为 B × 28 × 28 × 192。

所以结果可以记为：空间尺寸各减半，通道数翻倍。

## 7. 一个新 token 代表多大区域

最开始的 Patch Embedding 中，一个 token 对应原图的 4 × 4 区域。

第一次 Patch Merging 把 2 × 2 个相邻 tokens 合并，所以一个新 token 大致对应原图的 8 × 8 区域。

再次合并后，一个 token 对应的区域继续扩大。

这就是层级特征：浅层保留细小位置和纹理，深层逐渐组合成更大的部件和整体语义。

## 8. Patch Merging 与池化有什么不同

| 对比 | Patch Merging | 最大池化 |
|---|---|---|
| 怎样汇总 2 × 2 区域 | 拼接四个 tokens 后学习线性映射 | 每个通道选择最大值 |
| 是否有可学习参数 | 有 | 通常没有 |
| 通道数 | 通常翻倍 | 通常不变 |
| 是否保留四个位置来源 | 拼接阶段保留 | 只保留最大响应 |

二者都能下采样，但具体的信息汇总方式不同。

## 9. Patch Merging 位于哪里

Patch Merging 不放在同一个 Stage 的每个 Block 后面。

同一个 Stage 内，多个 Swin Blocks 保持 H、W、C 不变。完成整个 Stage 后，才使用一次 Patch Merging 进入下一个 Stage。

Swin-T 一共有四个 Stage，所以 Stage 之间共有三次 Patch Merging。Stage 4 后面不再合并，而是进入分类部分。

## 10. 本节小结

1. Patch Merging 每次合并空间上相邻的 2 × 2 tokens。
2. 四个 C 维 token 先拼成 4C 维，再通过线性层压成 2C 维。
3. 高和宽各减半，token 总数变成四分之一，通道数翻倍。
4. 新 token 对应更大的图片区域，使 Swin 形成多尺度层级特征。
5. Patch Merging 位于 Stage 之间，而不是每个 Swin Block 之后。

## 11. 自测问题

1. Patch Merging 为什么要取 2 × 2 个 tokens？
2. 四个 C 维 token 拼接后是多少维？
3. 为什么还要把 4C 压缩为 2C？
4. B × 56 × 56 × 96 经过 Patch Merging 后是什么 shape？
5. LayerNorm 会不会改变 shape？
6. Patch Merging 与最大池化有什么主要区别？
7. 为什么深层 token 能表示更大的图片区域？
8. 四个 Stage 之间一共有几次 Patch Merging？

### 自测参考答案

1. 这样能让高和宽各缩小一半，并汇总一个局部邻域。
2. 4C 维。
3. 防止通道扩大过快，并学习怎样提炼四个位置的信息。
4. B × 28 × 28 × 192。
5. 不会，只调整数值分布。
6. Patch Merging 拼接后使用可学习线性映射；最大池化直接选择最大值。
7. 每次合并都把相邻区域组合到一个新 token 中。
8. 三次。